# DIRECTER — Kaggle Inference

This notebook reproduces the supervisor's DIRECTER inference pipeline in a single Kaggle notebook.

**Only environment-level adaptations are made:**
- `torch.bfloat16` → `torch.float16`, as required for the Kaggle hardware/inference setup.
- Server-side input/output paths → Kaggle paths.
- The CLI runner is replaced by notebook cells; the underlying DIRECTER implementation is unchanged.
- The final predictions are saved as a single JSON file for later evaluation.

The supervisor's default model is `meta-llama/Llama-3.1-8B-Instruct`; that is the model used below.

## **Dependencies**

In [1]:
#!pip install -q -U transformers huggingface_hub

## **Imports**

In [2]:
from pathlib import Path
import os
import json
import time
import random
import numpy as np
import torch

from transformers import AutoModelForCausalLM, AutoTokenizer
from dataclasses import dataclass

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## **File Paths**

In [3]:
DATA_PATH = "/kaggle/input/datasets/anarvaaa/director-dataset/test.json"
OUTPUT_PATH = "/kaggle/working/directer_predictions.json"

### 1.1 `cache_ops.py`

In [4]:
DIRECTER_SRC = Path("/kaggle/working/directer_src")
DIRECTER_SRC.mkdir(parents=True, exist_ok=True)

Path(DIRECTER_SRC / "cache_ops.py").write_text(r'''"""In-place KV cache steering operations.

Key insight: Instruction tokens are processed during prefill and their KV vectors
never change during decoding. We snapshot only the instruction slices once, then
scale and restore them in place. No entire cache object is ever deep-copied.
"""

from typing import List, Tuple
import torch
from transformers import DynamicCache


# --- Version-tolerant cache accessors ---

def _keys(cache, layer: int) -> torch.Tensor:
    if hasattr(cache, "key_cache") and len(cache.key_cache) > 0:
        return cache.key_cache[layer]
    if hasattr(cache, "layers") and len(cache.layers) > 0:
        return cache.layers[layer].keys
    if isinstance(cache, (list, tuple)):
        return cache[layer][0]
    raise ValueError(f"Unsupported cache type: {type(cache)}")


def _values(cache, layer: int) -> torch.Tensor:
    if hasattr(cache, "value_cache") and len(cache.value_cache) > 0:
        return cache.value_cache[layer]
    if hasattr(cache, "layers") and len(cache.layers) > 0:
        return cache.layers[layer].values
    if isinstance(cache, (list, tuple)):
        return cache[layer][1]
    raise ValueError(f"Unsupported cache type: {type(cache)}")


def _set_keys(cache, layer: int, tensor: torch.Tensor) -> None:
    if hasattr(cache, "key_cache"):
        cache.key_cache[layer] = tensor
    elif hasattr(cache, "layers"):
        cache.layers[layer].keys = tensor
    elif isinstance(cache, list):
        cache[layer] = (tensor, cache[layer][1])


def _set_values(cache, layer: int, tensor: torch.Tensor) -> None:
    if hasattr(cache, "value_cache"):
        cache.value_cache[layer] = tensor
    elif hasattr(cache, "layers"):
        cache.layers[layer].values = tensor
    elif isinstance(cache, list):
        cache[layer] = (cache[layer][0], tensor)


def num_layers(cache) -> int:
    if hasattr(cache, "key_cache"):
        return len(cache.key_cache)
    if hasattr(cache, "layers"):
        return len(cache.layers)
    if isinstance(cache, (list, tuple)):
        return len(cache)
    raise ValueError(f"Unsupported cache type: {type(cache)}")


def _sync_seen(cache, length: int) -> None:
    if hasattr(cache, "_seen_tokens"):
        cache._seen_tokens = length


# --- Probe Cache Management ---

def pop_last_position(cache, keep_len: int):
    """Parks newly appended KV states from the raw pass while running probe passes."""
    saved = []

    for l in range(num_layers(cache)):
        k = _keys(cache, l)
        v = _values(cache, l)

        # Keep saved tensors on the same GPU as their corresponding layer.
        saved_k = k[:, :, keep_len:, :].clone()
        saved_v = v[:, :, keep_len:, :].clone()

        saved.append((saved_k, saved_v))

        _set_keys(
            cache,
            l,
            k[:, :, :keep_len, :]
        )
        _set_values(
            cache,
            l,
            v[:, :, :keep_len, :]
        )

    _sync_seen(cache, keep_len)
    return saved


def push_position(cache, saved) -> None:
    """Restores parked KV states back to the live cache."""

    for l, (k_new, v_new) in enumerate(saved):
        k_current = _keys(cache, l)
        v_current = _values(cache, l)

        # Explicitly place the parked tensors on the current layer's device.
        k_new = k_new.to(k_current.device)
        v_new = v_new.to(v_current.device)

        _set_keys(
            cache,
            l,
            torch.cat([k_current, k_new], dim=2)
        )

        _set_values(
            cache,
            l,
            torch.cat([v_current, v_new], dim=2)
        )

    _sync_seen(cache, _keys(cache, 0).shape[2])


def crop_to(cache, length: int) -> None:
    """Truncates trailing tokens appended during steering probes."""

    if hasattr(cache, "crop"):
        try:
            cache.crop(length)
            return
        except Exception:
            pass

    for l in range(num_layers(cache)):
        _set_keys(
            cache,
            l,
            _keys(cache, l)[:, :, :length, :]
        )
        _set_values(
            cache,
            l,
            _values(cache, l)[:, :, :length, :]
        )

    _sync_seen(cache, length)


# --- In-Place Span Steerer ---

class SpanSteerer:
    """Scales key vectors of the instruction span in-place on selected layers."""

    def __init__(
        self,
        cache,
        token_spans: List[Tuple[int, int]],
        key_scale: float,
        value_scale: float = 1.0,
    ):
        self.cache = cache
        self.key_scale = key_scale
        self.value_scale = value_scale
        self.n_layers = num_layers(cache)

        idx = sorted({
            i
            for s, e in token_spans
            for i in range(s, e)
        })

        # Index itself is kept as a CPU tensor. It is moved to the
        # corresponding layer's GPU immediately before indexing.
        self.index = torch.tensor(
            idx,
            dtype=torch.long
        )

        # Snapshot the original unscaled slices from prefill.
        # Each layer's snapshot remains on that layer's GPU.
        self._orig_k = [
            _keys(cache, l)[
                :,
                :,
                self.index.to(_keys(cache, l).device),
                :
            ].clone()
            for l in range(self.n_layers)
        ]

        self._orig_v = (
            [
                _values(cache, l)[
                    :,
                    :,
                    self.index.to(_values(cache, l).device),
                    :
                ].clone()
                for l in range(self.n_layers)
            ]
            if value_scale != 1.0
            else None
        )

        self._dirty: List[int] = []

    def apply(self, layers: List[int]) -> None:
        """Applies key scaling to the specified layers in place."""

        self.restore()

        for l in layers:
            keys = _keys(self.cache, l)
            idx = self.index.to(keys.device)

            keys[:, :, idx, :] = (
                self._orig_k[l] * self.key_scale
            )

            if self._orig_v is not None:
                values = _values(self.cache, l)
                idx_v = self.index.to(values.device)

                values[:, :, idx_v, :] = (
                    self._orig_v[l] * self.value_scale
                )

        self._dirty = list(layers)

    def restore(self) -> None:
        """Restores scaled layers back to their original unscaled slices."""

        for l in self._dirty:
            keys = _keys(self.cache, l)
            idx = self.index.to(keys.device)

            keys[:, :, idx, :] = self._orig_k[l]

            if self._orig_v is not None:
                values = _values(self.cache, l)
                idx_v = self.index.to(values.device)

                values[:, :, idx_v, :] = self._orig_v[l]

        self._dirty = []
''', encoding="utf-8")

print("Wrote", DIRECTER_SRC / "cache_ops.py")

Wrote /kaggle/working/directer_src/cache_ops.py


### 1.2 `guidelines.py`

In [5]:
DIRECTER_SRC = Path("/kaggle/working/directer_src")
DIRECTER_SRC.mkdir(parents=True, exist_ok=True)

Path(DIRECTER_SRC / "guidelines.py").write_text('"""Annotation guideline text, examples removed.\n\nOnly the explicit *rules* live here. The worked examples from the guideline\ndocument are deliberately excluded to prevent bloat in the steered KV cache.\n"""\n\nGENERAL_RULES = """General rules:\n1. Always include the article words before the entities - \'the\', \'a\', \'an\'. All markable concepts should be derived from noun phrases.\n2. Include special characters.\n3. Only the Title and the Methods section are to be annotated."""\n\n\nELEMENTS = {\n    "Population": {\n        "letter": "P",\n        "name": "Participant",\n        "definition": (\n            "The Participant component refers to the descriptions of participants involved in a "\n            "medical study, including their recruitment process and the characteristics or "\n            "requirements they needed to meet for inclusion, such as age, gender, sample size, "\n            "medical diagnoses or conditions, and treatment locations."\n        ),\n        "rules": """What to include:\n1. Mark the longest noun phrases about how the participants were recruited in the medical study and what characteristics or requirements they needed to meet to be included in the study.\n2. When subgroup information is inevitably included in the complete noun phrase, include it.\n\nWhat to exclude:\n1. Phrases about grouping should not be considered as Participant.\n2. When a location is mentioned together with participants, include them. Otherwise, do not include them.\n3. Do not annotate inclusion or exclusion criteria for the participants when the phrases "inclusion criteria" or "exclusion criteria" are explicitly mentioned.\n4. Do not include phrases that only mention a condition without participants being mentioned.\n5. Exclude prepositional words before the participants.""",\n    },\n\n    "Intervention": {\n        "letter": "I",\n        "name": "Intervention",\n        "definition": (\n            "The Intervention component represents the proposed treatment or approach being "\n            "administered to the participants. Interventions are not limited to medical "\n            "treatments; they may also be non-medical approaches such as educational courses "\n            "or therapies."\n        ),\n        "rules": """What to include:\n1. Include all the modifiers before the Intervention element when dosage, frequency, and the form of the intervention appear as modifiers.\n2. Include an Intervention entity when it is inside a prepositional phrase.\n3. If there are multiple distinct entities mentioned within the same context, split them into separate annotations.\n4. Annotate continuous entities separately.\n5. Include all the general and specific terms of the intervention.\n6. Annotate the intervention every time it appears, whether or not it has already been annotated.\n7. When one drug is co-administered with another, annotate them separately.\n\nWhat to exclude:\n1. Do not include general terms like \'control group\', \'intervention group\', or \'a treated group\'.\n2. When an intervention is mentioned as a group, annotate only the intervention itself; do not include the word \'group\'.\n3. Exclude dosage, frequency, length, and the form of the intervention when they are located after the Intervention element.\n4. Do not include any prepositional phrase following the Intervention element.""",\n    },\n\n    "Comparator": {\n        "letter": "C",\n        "name": "Control",\n        "definition": (\n            "The Control component refers to the comparison or control treatment used in the "\n            "study. The control group serves as a baseline against which the effectiveness of "\n            "the intervention is evaluated. It may receive a placebo, a sham procedure, or no "\n            "treatment at all."\n        ),\n        "rules": """What to include:\n1. All the treatments or procedures that are given to all the patients in the control group should be annotated as Control.\n2. Control entities follow exactly the same rules as Intervention entities. The only difference is that Control is semantically the comparator treatment.\n3. Control and Intervention may sometimes be the same treatment. In these cases, annotate them according to semantics: the one given to the study group is Intervention, and the one given to the control group is Control.\n\nWhat to exclude:\n1. Do not include general terms like \'control group\' or \'a treated group\'.\n2. When a control is mentioned as a group, annotate only the treatment itself; do not include the word \'group\'.\n3. Exclude dosage, frequency, length, and the form of the treatment when they are located after the Control element.\n4. Do not include any prepositional phrase following the Control element.""",\n    },\n\n    "Outcome": {\n        "letter": "O",\n        "name": "Outcome",\n        "definition": (\n            "The Outcome component refers to the measurements or observations used to assess the "\n            "effectiveness of the treatment in individuals participating in a trial. Outcomes are "\n            "often described by the specific score, scale, measurement tool, or clinical test used "\n            "to evaluate the desired result."\n        ),\n        "rules": """What to include:\n1. If a clinical test is the main part of the study and is being used as an intervention, then it has to be annotated as Intervention, not Outcome.\n2. Include the outcomes separately when their abbreviations and expansions are both mentioned.\n3. Include an outcome that is illustrated both at a summative level and at a detailed level.\n4. Include up to one prepositional phrase.\n5. If an affirmative sentence is provided, the outcome should be what suggests the result of the study.\n6. Include words indicating statistical measurement such as "log", "mean", and "proportion" before the outcome.\n7. Include the words "a composite of" when two outcomes are combined.\n\nWhat to exclude:\n1. Do not include how the outcome was measured, unless it is a modifier.\n2. Do not include the time span or time point at which the outcome was measured, unless it is a modifier or the measurement depends on time.\n3. Do not include how often the outcome was measured, unless it is a modifier.\n4. Do not include who received the outcome measure.\n5. Do not include information about how an outcome is compared.\n6. Do not include the word that indicates the difference, such as "change" or "difference".\n7. Do not include statistical methods.\n8. Do not annotate outcomes in titles.""",\n    },\n}\n\nELEMENT_ORDER = ["Population", "Intervention", "Comparator", "Outcome"]', encoding="utf-8")
print("Wrote", DIRECTER_SRC / "guidelines.py")

Wrote /kaggle/working/directer_src/guidelines.py


### 1.3 `prompting.py`

In [6]:
DIRECTER_SRC = Path("/kaggle/working/directer_src")
DIRECTER_SRC.mkdir(parents=True, exist_ok=True)

Path(DIRECTER_SRC / "prompting.py").write_text('"""Prompt construction with exact steered-span localisation.\n\nThe span is located by character offsets rather than by re-tokenising the\ninstruction text and searching for a matching id sequence. Standalone\nre-tokenisation drifts from in-context tokenisation (leading-space merges,\nnewlines, punctuation), and on a multi-hundred-token rules block it can fail\nsilently and return an empty span.\n"""\n\nfrom typing import List, Tuple\n\nfrom guidelines import ELEMENTS, GENERAL_RULES\n\n\n# Sentinels: unique strings stripped out before the prompt is tokenised\n_OPEN = "<<<STEER>>>"\n_CLOSE = "<<</STEER>>>"\n\n\ndef _build_user_message(abstract: str, element: str, cfg) -> str:\n    """Task first, output format second, steered instruction block last."""\n    meta = ELEMENTS[element]\n\n    task = (\n        f"You are a biomedical annotator extracting the {meta[\'name\']} "\n        f"({meta[\'letter\']}) elements from the clinical trial abstract below.\\n\\n"\n        f"Abstract:\\n{abstract.strip()}\\n"\n    )\n\n    fmt = (\n        f\'\\nReturn a single JSON object with exactly one key "{element}", whose value is a \'\n        f"list of strings. Each string must be a span copied verbatim from the abstract. "\n        f"Use an empty list if no such element is present. Output the JSON object only, "\n        f"with no explanation before or after it.\\n"\n    )\n\n    rules = f"{meta[\'definition\']}\\n\\n{meta[\'rules\']}"\n    if cfg.steer_general_rules:\n        rules = f"{GENERAL_RULES}\\n\\n{rules}"\n\n    instruction = (\n        "\\nYour response must follow the annotation guidelines below:\\n"\n        f"{_OPEN}{rules}{_CLOSE}\\n"\n    )\n\n    if cfg.steer_output_format:\n        return task + f"{_OPEN}{fmt}{_CLOSE}" + instruction.replace(_OPEN, "").replace(_CLOSE, "")\n    return task + fmt + instruction\n\n\ndef build_element_prompt(abstract: str, element: str, tokenizer, cfg):\n    """Builds chat prompt and returns (input_ids, [(start_token, end_token_exclusive)])."""\n    user_msg = _build_user_message(abstract, element, cfg)\n\n    # Render chat template to string so we can measure character offsets\n    templated = tokenizer.apply_chat_template(\n        [{"role": "user", "content": user_msg}],\n        tokenize=False,\n        add_generation_prompt=True,\n    )\n\n    # Strip sentinels while recording exact character start and end positions\n    clean_parts, char_spans, cursor = [], [], 0\n    text, out_len = templated, 0\n    while True:\n        i = text.find(_OPEN, cursor)\n        if i == -1:\n            clean_parts.append(text[cursor:])\n            break\n        j = text.find(_CLOSE, i)\n        if j == -1:\n            raise ValueError("Unbalanced steering sentinel in prompt.")\n        head = text[cursor:i]\n        body = text[i + len(_OPEN):j]\n        clean_parts.append(head)\n        out_len += len(head)\n        char_spans.append((out_len, out_len + len(body)))\n        clean_parts.append(body)\n        out_len += len(body)\n        cursor = j + len(_CLOSE)\n\n    clean = "".join(clean_parts)\n\n    # Fast tokenizer returns character-to-token offset mappings\n    enc = tokenizer(\n        clean,\n        add_special_tokens=False,\n        return_offsets_mapping=True,\n        return_tensors="pt",\n    )\n    offsets = enc["offset_mapping"][0].tolist()\n\n    # Map character spans to token indices\n    token_spans: List[Tuple[int, int]] = []\n    for cs, ce in char_spans:\n        idx = [\n            t for t, (s, e) in enumerate(offsets)\n            if s < ce and e > cs and e > s  # Overlapping non-empty tokens\n        ]\n        if not idx:\n            raise ValueError(f"Could not localise steered span for \'{element}\'.")\n        token_spans.append((idx[0], idx[-1] + 1))  # End-exclusive token index\n\n    total = enc["input_ids"].shape[1]\n    # Prefill runs on input_ids[:, :-1], so the steered span must reside inside prefill\n    if max(e for _, e in token_spans) > total - 1:\n        raise ValueError("Steered span runs into the last prompt token.")\n\n    if cfg.verbose:\n        covered = sum(e - s for s, e in token_spans)\n        print(f"[{element}] prompt={total} tok, steered={covered} tok, spans={token_spans}")\n\n    return enc["input_ids"], token_spans', encoding="utf-8")
print("Wrote", DIRECTER_SRC / "prompting.py")

Wrote /kaggle/working/directer_src/prompting.py


### 1.4 `directer.py`

In [7]:
DIRECTER_SRC = Path("/kaggle/working/directer_src")
DIRECTER_SRC.mkdir(parents=True, exist_ok=True)

Path(DIRECTER_SRC / "directer.py").write_text(r'''"""DIRECTER: KV-cache steering with attention-sensitivity layer ranking and a
plausibility-guided decoding loop.

Faithful to Algorithm 1 and Algorithm 2 of Kang & Kim (ICLR 2026).
"""

from typing import List, Optional, Tuple

import torch
import torch.nn.functional as F
from transformers import DynamicCache

from cache_ops import SpanSteerer, crop_to, pop_last_position, push_position


class DirecterGenerator:
    def __init__(self, model, tokenizer, cfg):
        self.model = model
        self.tokenizer = tokenizer
        self.cfg = cfg
        self.device = model.device
        self.n_layers = model.config.num_hidden_layers
        self.all_layers = list(range(self.n_layers))

        # Handle both standard EOS and Llama-3's turn terminator <|eot_id|>
        stops = {tokenizer.eos_token_id}
        eot = tokenizer.convert_tokens_to_ids("<|eot_id|>")
        if isinstance(eot, int) and eot >= 0:
            stops.add(eot)
        self.stop_ids = {s for s in stops if s is not None}

        self._states: List[Tuple[torch.Tensor, torch.Tensor]] = []
        self._hooks = []

    # ---------------- Attention Hooks ----------------

    def _hook_fn(self, module, args, kwargs, result):
        # H_pre  = module input hidden state
        # H_post = module output hidden state
        pre = kwargs.get("hidden_states", args[0] if args else None)
        post = result[0] if isinstance(result, tuple) else result
        if pre is not None and post is not None:
            self._states.append((pre.detach().float(), post.detach().float()))

    def _attach_hooks(self):
        self._states = []
        for name, mod in self.model.named_modules():
            if "layers." in name and name.endswith(".self_attn"):
                self._hooks.append(
                    mod.register_forward_hook(self._hook_fn, with_kwargs=True)
                )

    def _detach_hooks(self):
        for h in self._hooks:
            h.remove()
        self._hooks.clear()
        self._states = []

    # ---------------- Plausibility Machinery ----------------

    def _steering_can_matter(self, logits: torch.Tensor) -> bool:
        """The paper's gating condition (Section 3.2).

        If raw top-2 probability < beta * top-1 probability, no steered distribution
        could satisfy the plausibility constraint except when top-1 is unchanged.
        """
        probs = torch.softmax(logits[:, -1, :].float(), dim=-1)
        top2 = probs.topk(2, dim=-1).values.flatten()
        return bool(top2[1] >= self.cfg.plausibility * top2[0])

    def _is_plausible(self, raw_logits: torch.Tensor, steered_logits: torch.Tensor) -> bool:
        """Accepts iff p_raw[steered_top1] >= beta * p_raw[raw_top1] (Eq. 1)."""
        raw = raw_logits[:, -1, :].float()
        steered = steered_logits[:, -1, :].float()
        p_raw = F.softmax(raw, dim=-1)

        i_steered = steered.argmax(dim=-1)
        i_raw = raw.argmax(dim=-1)
        p_of_steered = p_raw.gather(-1, i_steered.unsqueeze(-1)).squeeze(-1)
        p_of_raw = p_raw.gather(-1, i_raw.unsqueeze(-1)).squeeze(-1)
        return bool((p_of_steered >= self.cfg.plausibility * p_of_raw).item())

    @staticmethod
    def _halve(ranked: List[int], trial: int) -> List[int]:
        """Keeps the top |L| / (2**trial) most sensitive layers."""
        keep = len(ranked) // (2 ** trial)
        return ranked[:keep]

    # ---------------- Layer Ranking ----------------

    def _rank_layers(self, token_id, cache, steerer, base_len, raw_states) -> List[int]:
        """One-time attention-sensitivity ranking (Eq. 2 & Eq. 6 in the paper).

        Sensitivity(l) ∝ sum_j [ dist(H_pre^j, H_post^{j,l}) + dist(H_pre^{j,l}, H_post^j) ]
        """
        rows = []
        for probe in self.all_layers:
            steerer.apply([probe])
            self._states = []
            self.model(input_ids=token_id, past_key_values=cache, use_cache=True)
            steered_states = self._states
            steerer.restore()
            crop_to(cache, base_len)

            per_layer = []
            for j in range(self.n_layers):
                h_pre, h_post = raw_states[j]
                s_pre, s_post = steered_states[j]

                # With device_map="auto", different transformer layers may live
                # on different GPUs. Perform each layer's comparison locally.
                layer_device = h_pre.device
                h_post = h_post.to(layer_device)
                s_pre = s_pre.to(layer_device)
                s_post = s_post.to(layer_device)

                d1 = 1.0 - F.cosine_similarity(h_pre, s_post, dim=-1)
                d2 = 1.0 - F.cosine_similarity(s_pre, h_post, dim=-1)

                # Only the resulting scalar is moved to GPU 0 for aggregation.
                per_layer.append((d1 + d2).mean().to("cuda:0"))

            rows.append(torch.stack(per_layer).mean())

        scores = torch.stack(rows)  # [L]
        order = torch.argsort(scores, descending=True)  # Most sensitive layers first
        ranked = [int(i) for i in order.tolist()]

        if self.cfg.verbose:
            print(f"  [DIRECTER] Top-8 sensitive layers: {ranked[:8]}")
        return ranked

    # ---------------- Autoregressive Generation ----------------

    @torch.inference_mode()
    def generate(self, input_ids: torch.Tensor, token_spans: List[Tuple[int, int]]) -> str:
        input_ids = input_ids.to(self.device)

        # Prefill prompt up to token T-1
        out = self.model(input_ids[:, :-1], use_cache=True, return_dict=True)
        cache = out.past_key_values

        # Convert legacy tuple cache to HuggingFace DynamicCache
        if isinstance(cache, tuple):
            cache = DynamicCache.from_legacy_cache(cache)

        next_token = input_ids[:, -1:]

        steerer = SpanSteerer(
            cache,
            token_spans,
            key_scale=self.cfg.key_scale_factor,
            value_scale=self.cfg.value_scale_factor,
        )

        ranked: Optional[List[int]] = None
        generated: List[int] = []
        self._attach_hooks()

        try:
            for _ in range(self.cfg.max_new_tokens):
                base_len = cache.get_seq_length()

                # 1. Raw forward pass (advances KV cache)
                self._states = []
                raw_out = self.model(input_ids=next_token, past_key_values=cache, use_cache=True)
                raw_logits = raw_out.logits
                raw_states = self._states if ranked is None else None

                need_rank = ranked is None
                need_probe = self._steering_can_matter(raw_logits)
                final_logits = raw_logits

                if need_rank or need_probe:
                    # Park the newly appended KV so probes evaluate on the pre-step state
                    parked = pop_last_position(cache, base_len)

                    # Step 1: Run one-time layer ranking and immediately drop hooks
                    if need_rank:
                        ranked = self._rank_layers(next_token, cache, steerer, base_len, raw_states)
                        self._detach_hooks()

                    # Plausibility-guided halving loop
                    if need_probe:
                        candidates = list(ranked)
                        trial = 0
                        while candidates and trial <= self.cfg.max_trial:
                            steerer.apply(candidates)
                            steered_logits = self.model(
                                input_ids=next_token,
                                past_key_values=cache,
                                use_cache=True,
                            ).logits
                            steerer.restore()
                            crop_to(cache, base_len)

                            if self._is_plausible(raw_logits, steered_logits):
                                final_logits = steered_logits
                                break

                            trial += 1
                            candidates = self._halve(ranked, trial)

                    push_position(cache, parked)

                next_token = final_logits[:, -1, :].argmax(dim=-1, keepdim=True)
                tid = int(next_token.item())
                if tid in self.stop_ids:
                    break
                generated.append(tid)
        finally:
            self._detach_hooks()

        return self.tokenizer.decode(generated, skip_special_tokens=True)
''', encoding="utf-8")

print("Wrote", DIRECTER_SRC / "directer.py")

Wrote /kaggle/working/directer_src/directer.py


### 1.5 `pipeline.py`

In [8]:
DIRECTER_SRC = Path("/kaggle/working/directer_src")
DIRECTER_SRC.mkdir(parents=True, exist_ok=True)

Path(DIRECTER_SRC / "pipeline.py").write_text('"""PICO extraction pipeline: 4 steered runs per abstract merged into one JSON."""\n\nimport json\nfrom typing import Dict, List\n\nfrom directer import DirecterGenerator\nfrom guidelines import ELEMENT_ORDER\nfrom prompting import build_element_prompt\n\n\ndef _extract_json_object(text: str) -> dict:\n    """Extracts the first balanced JSON object {...} from raw model output."""\n    start = text.find("{")\n    if start == -1:\n        return {}\n    depth, in_str, escaped = 0, False, False\n    for i in range(start, len(text)):\n        ch = text[i]\n        if in_str:\n            if escaped:\n                escaped = False\n            elif ch == "\\\\":\n                escaped = True\n            elif ch == \'"\':\n                in_str = False\n            continue\n        if ch == \'"\':\n            in_str = True\n        elif ch == "{":\n            depth += 1\n        elif ch == "}":\n            depth -= 1\n            if depth == 0:\n                try:\n                    return json.loads(text[start : i + 1])\n                except json.JSONDecodeError:\n                    return {}\n    return {}\n\n\ndef _coerce_spans(value) -> List[str]:\n    """Normalizes extracted model values into a clean list of unique string spans."""\n    if value is None:\n        return []\n    if isinstance(value, str):\n        value = [value]\n    if not isinstance(value, list):\n        return []\n\n    out, seen = [], set()\n    for item in value:\n        if not isinstance(item, str):\n            continue\n        s = item.strip()\n        if s and s.lower() not in seen:\n            seen.add(s.lower())\n            out.append(s)\n    return out\n\n\nclass PICODirecterPipeline:\n    def __init__(self, model, tokenizer, cfg):\n        self.tokenizer = tokenizer\n        self.cfg = cfg\n        self.generator = DirecterGenerator(model, tokenizer, cfg)\n\n    def extract(self, abstract: str, return_raw: bool = False) -> Dict:\n        result, raw = {}, {}\n\n        for element in ELEMENT_ORDER:\n            input_ids, spans = build_element_prompt(\n                abstract, element, self.tokenizer, self.cfg\n            )\n            text = self.generator.generate(input_ids, spans)\n            raw[element] = text\n\n            obj = _extract_json_object(text)\n            # Accept exact key or single-key fallback\n            value = obj.get(element)\n            if value is None and len(obj) == 1:\n                value = next(iter(obj.values()))\n\n            result[element] = _coerce_spans(value)\n\n            if self.cfg.verbose:\n                print(f"[{element}] -> Found {len(result[element])} span(s)")\n\n        return {"pico": result, "raw": raw} if return_raw else result', encoding="utf-8")
print("Wrote", DIRECTER_SRC / "pipeline.py")

Wrote /kaggle/working/directer_src/pipeline.py


## **Configurations**

**Initial Config - alpha = 100, beta = 0.5**

Alpha values to test - 50, 75, 100, 125, 150

Beta values to test  - 0.3, 0.4, 0.5, 0.6, 0.7

**Initial (Alpha, Beta) Pairs ->**

(50, 0.3), (50,0.5), (50, 0.7)

(100, 0.3), (100. 0.5), (100, 0.7)

(150, 0.3), (150, 0.5), (150, 0.7)

In [9]:
MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
DEVICE = "cuda"
ATTN_IMPLEMENTATION = "sdpa"

ALPHA = 100.0
BETA = 0.5
MAX_TRIAL = 10
MAX_NEW_TOKENS = 512

STEER_GENERAL_RULES = True
STEER_OUTPUT_FORMAT = False
VERBOSE = False

## **DIRECTER configuration object**

COnsidering key_scale factor = 1.0 (default), 0.5 and 1.5

In [10]:
@dataclass
class DirecterConfig:
    model_id: str = MODEL_NAME
    device: str = DEVICE
    attn_implementation: str = ATTN_IMPLEMENTATION
    key_scale_factor: float = ALPHA
    value_scale_factor: float = 0.5
    plausibility: float = BETA
    max_trial: int = MAX_TRIAL
    max_new_tokens: int = MAX_NEW_TOKENS
    steer_general_rules: bool = STEER_GENERAL_RULES
    steer_output_format: bool = STEER_OUTPUT_FORMAT
    verbose: bool = VERBOSE

cfg = DirecterConfig()
print(cfg)

DirecterConfig(model_id='meta-llama/Llama-3.1-8B-Instruct', device='cuda', attn_implementation='sdpa', key_scale_factor=100.0, value_scale_factor=0.5, plausibility=0.5, max_trial=10, max_new_tokens=512, steer_general_rules=True, steer_output_format=False, verbose=False)


## **Hugging Face authentication**

In [11]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("my_hf_token")
login(token=hf_token)

## **Load the model and tokenizer**

This follows the supervisor's `model_io.py` loading configuration. The only hardware adaptation is `torch.float16` instead of `torch.bfloat16`.

In [12]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=hf_token,
    use_fast=True,
)
if not tokenizer.is_fast:
    raise RuntimeError("A fast tokenizer is required for offset mapping in span localisation.")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=hf_token,
    device_map="auto",
    dtype=torch.float16,     # torch_dtype is depreciated
    low_cpu_mem_usage=True,
    attn_implementation=ATTN_IMPLEMENTATION,
)
model.eval()

if getattr(model, "generation_config", None) is not None:
    model.generation_config.update(
        do_sample=False,
        temperature=None,
        top_p=None,
        top_k=None,
    )

print(f"Loaded {MODEL_NAME}")
print(f"Layers: {model.config.num_hidden_layers}")
print(f"Hidden size: {model.config.hidden_size}")
print(f"Model device: {model.device}")

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Loaded meta-llama/Llama-3.1-8B-Instruct
Layers: 32
Hidden size: 4096
Model device: cuda:0


## **Import the unchanged DIRECTER implementation**

In [13]:
import sys
sys.path.insert(0, str(DIRECTER_SRC))

from cache_ops import SpanSteerer, crop_to, pop_last_position, push_position
from guidelines import GENERAL_RULES, ELEMENTS, ELEMENT_ORDER
from prompting import build_element_prompt
from directer import DirecterGenerator
from pipeline import PICODirecterPipeline

print("DIRECTER implementation imported successfully.")

DIRECTER implementation imported successfully.


## **Load `test.json` from the Kaggle dataset**

In [14]:
with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

if not isinstance(data, list):
    raise TypeError(f"Expected test.json to contain a JSON list, got {type(data).__name__}")

print("Number of records:", len(data))
print("First record keys:", list(data[0].keys()) if data else [])

Number of records: 250
First record keys: ['title', 'abstract', 'population', 'intervention_exposure', 'comparison', 'outcome']


## **Create the pipeline**

In [15]:
pipe = PICODirecterPipeline(model, tokenizer, cfg)
print("Pipeline ready.")

Pipeline ready.


## **Run inference**

This reproduces the supervisor's `run.py` loop: each record is passed to the PICO DIRECTER pipeline, and the four elements are extracted separately and merged into one result.

`save_raw` is kept **off**, matching the supervisor's default runner behavior, so the saved file contains the original record plus `pico_directer`.

In [16]:
# Test DIRECTER on a single example before running the full dataset
with open(DATA_PATH, "r") as f:
    test_data = json.load(f)

print("Number of examples:", len(test_data))

example = test_data[1]
abstract = example["abstract"]

print("Testing first example...")
print("Abstract length:", len(abstract))

result = pipe.extract(abstract, return_raw=True)

print("\nResult:")
print(result)

Number of examples: 250
Testing first example...
Abstract length: 653

Result:
{'pico': {'Population': ['Ten healthy volunteers', 'healthy volunteers'], 'Intervention': ['an infusion of saline', 'remifentanil', 'alfentanil'], 'Comparator': ['saline', 'a placebo'], 'Outcome': ['mood forms', 'psychomotor tests', 'miosis', 'analgesia', 'a cold-pressor test']}, 'raw': {'Population': '{\n  "Population": [\n    "Ten healthy volunteers",\n    "healthy volunteers"\n  ]\n}', 'Intervention': '{\n  "Intervention": [\n    "an infusion of saline",\n    "remifentanil",\n    "alfentanil",\n    "an infusion of saline",\n    "remifentanil",\n    "alfentanil"\n  ]\n}', 'Comparator': '{\n  "Comparator": [\n    "saline",\n    "a placebo"\n  ]\n}', 'Outcome': '{\n  "Outcome": [\n    "mood forms",\n    "psychomotor tests",\n    "miosis",\n    "analgesia",\n    "a cold-pressor test"\n  ]\n}'}}


In [ ]:
results = []
t0 = time.time()

for n, row in enumerate(data[:61], 1):
    abstract_text = row.get("abstract") or row.get("text", "")

    if not abstract_text:
        continue

    res = pipe.extract(abstract_text, return_raw=False)

    payload = dict(row)
    payload["pico_directer"] = res
    results.append(payload)

    elapsed = time.time() - t0
    avg_per_doc = elapsed / n

    print(
        f"[{n}/{len(data[:61])}] "
        f"ID: {row.get('id', 'N/A')} "
        f"(Avg: {avg_per_doc:.1f}s/doc)"
    )

print("=" * 60)
print(f"Finished: {len(results)} records")
print(f"Total time: {time.time() - t0:.1f}s")

[1/61] ID: N/A (Avg: 28.2s/doc)
[2/61] ID: N/A (Avg: 27.4s/doc)
[3/61] ID: N/A (Avg: 26.8s/doc)
[4/61] ID: N/A (Avg: 28.3s/doc)
[5/61] ID: N/A (Avg: 27.0s/doc)
[6/61] ID: N/A (Avg: 26.3s/doc)
[7/61] ID: N/A (Avg: 26.0s/doc)
[8/61] ID: N/A (Avg: 27.5s/doc)
[9/61] ID: N/A (Avg: 27.9s/doc)
[10/61] ID: N/A (Avg: 27.4s/doc)
[12/61] ID: N/A (Avg: 28.8s/doc)


In [18]:
'''
empty_indices = []

for n, row in enumerate(data, 1):
    abstract_text = row.get("abstract") or row.get("text", "")

    if not abstract_text:
        empty_indices.append(n)

print("Empty/missing abstract indices:")
print(empty_indices)
print("Count:", len(empty_indices))
'''

'\nempty_indices = []\n\nfor n, row in enumerate(data, 1):\n    abstract_text = row.get("abstract") or row.get("text", "")\n\n    if not abstract_text:\n        empty_indices.append(n)\n\nprint("Empty/missing abstract indices:")\nprint(empty_indices)\nprint("Count:", len(empty_indices))\n'

In [19]:
'''
missing = [11, 23, 26, 28, 31, 32, 38, 40, 41, 47,
49, 96, 115, 135, 136, 139, 149, 153, 154, 169,
172, 180, 204, 214, 216, 217, 218, 227, 231, 233,
234, 236, 237, 239, 242] 
'''

'\nmissing = [11, 23, 26, 28, 31, 32, 38, 40, 41, 47,\n49, 96, 115, 135, 136, 139, 149, 153, 154, 169,\n172, 180, 204, 214, 216, 217, 218, 227, 231, 233,\n234, 236, 237, 239, 242] \n'

## **Save predictions as JSON**

The supervisor saved JSONL. For the Kaggle workflow, this saves the same prediction payloads as a standard JSON array, which can be loaded directly later for evaluation.

In [20]:
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"Saved {len(results)} records to:")
print(OUTPUT_PATH)

with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
    saved = json.load(f)

print("Reloaded successfully:", len(saved), "records")
if saved:
    print("Prediction keys:", saved[0].get("pico_directer", {}).keys())

Saved 50 records to:
/kaggle/working/directer_predictions.json
Reloaded successfully: 50 records
Prediction keys: dict_keys(['Population', 'Intervention', 'Comparator', 'Outcome'])


## **Inspect one prediction**

In [22]:
if results:
    print(json.dumps(results[0], ensure_ascii=False, indent=2))
else:
    print("No predictions were produced.")

{
  "title": "Switching patients with asthma from chlorofluorocarbon (CFC) albuterol to hydrofluoroalkane-134a (HFA) albuterol.",
  "abstract": "METHODS: Patients with asthma stabilized on CFC albuterol during a 12-week safety and efficacy trial were randomized to either continue receiving CFC albuterol or to be switched to receive HFA albuterol in a yearlong safety and efficacy trial. Safety and efficacy were compared over the first 12 weeks of the yearlong trial between patients who had remained on CFC albuterol and those who had been switched to HFA albuterol. Bronchodilator efficacy was evaluated by serial spirometry for 6 hr after the patients self-administered the study drug in the clinic. Safety was assessed by measuring changes in pulse rate, blood pressure, and electrocardiogram (ECG) intervals after dosing with study drug, monitoring adverse events, and performing prestudy and poststudy laboratory testing and physical examinations.",
  "population": [
    "patients with asthm

## 13. Optional: release model memory after inference

In [ ]:
# Optional: uncomment before evaluation if GPU memory needs to be released.
# import gc
# del pipe, model, tokenizer
# gc.collect()
# torch.cuda.empty_cache()